# 🕵️ Dia 90 — Portfólio, currículo e onde se candidatar

**Investigador Cyber 90 Dias** · Fase 7 · Rank Jutsu `S`

> Baseado no *Day 90* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-7-capstone-career/day-90.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).
> **Material sintético.** O *case packet* (Orrin Valley / Castellan) e a Operação Isca são ficção; domínios `.example`, IPs de documentação.

---

### 🎬 Onde paramos

A Marta agradeceu por você ter escrito *"o que sabia, e só o que sabia"*. Rafa trouxe dois cafés e uma pergunta: *"Daqui a um ano, numa entrevista, o que você vai mostrar? Nada da Operação Isca sai daqui. O portfólio mostra o método, nunca o cliente."*

Neste notebook você vai:

1. **Rodar as checagens de publicação** do original sobre uma pasta de portfólio de exemplo.
2. **Ver as revisões antigas de um PDF** sem exiftool (Checkpoint 15).
3. **Reescrever tópicos de currículo** como ação + ferramenta + resultado.
4. **Montar o rastreador de alvos** e contar as lacunas.
5. **Jogar o quiz função × artefato.**
6. **Fechar a Operação Isca.**

🐍 Só biblioteca padrão do Python. Funciona offline: se a rede falhar, o notebook usa a cópia embutida dos arquivos. Rode as células de cima para baixo (Shift+Enter).

## 1. Uma pasta de portfólio de exemplo (com erros plantados)

Para treinar as checagens, o notebook cria uma pasta `investigation-portfolio/` pequena, com **três erros de propósito**: um IP não desarmado, um arquivo sem rótulo de sintético e um nome de pessoa real fictício-mas-plausível num laboratório público. Você vai achá-los com código.

In [ ]:
from pathlib import Path

PORT = Path("investigation-portfolio")
ARQUIVOS = {
    "README.md": "# Portfólio de investigação\n\nSYNTHETIC. Mirando CTI e fraude. Para CTI, abra capstone-castellan-bec/.\n"
                 "Referência: https://attack.mitre.org/techniques/T1564/008/\n",
    "capstone-castellan-bec/README.md": "# Capstone (synthetic BEC case)\n\nSYNTHETIC TRAINING CASE.\n"
                 "Submitter IP 198.51.100[.]23; phishing hxxps://cstl-docshare[.]example/view/r\n",
    "capstone-castellan-bec/indicators.csv": "indicator,type\n198.51.100.23,ip\ncastellan-hardwood[.]example,domain\n",   # erro 1 e 2
    "fraud-ring-graph/README.md": "# Fraud ring graph (synthetic)\n\nSYNTHETIC. O perfil público de Fulano de Tal Exemplo aparecia no grafo.\n",  # erro 3
    "operacao-isca-sintetica/README.md": "# Operação Isca (FICTÍCIO)\n\nCaso fictício. Domínio da isca: serraazul-pix[.]example\n",
}
for nome, texto in ARQUIVOS.items():
    p = PORT / nome
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(texto, encoding="utf-8")
print("\n".join(str(p) for p in sorted(PORT.rglob("*")) if p.is_file()))

## 2. As checagens 2, 3 e 4 em Python

A checagem 3 é o `grep -rnE` do original, com a mesma expressão regular. Links para páginas de referência reais são permitidos: o notebook mantém uma lista de domínios de referência para não te alarmar à toa.

In [ ]:
import re

PADRAO_GREP = re.compile(r"([0-9]{1,3}\.){3}[0-9]{1,3}|https?://|[a-z0-9-]+\.example")   # a mesma do original
REFERENCIAS_OK = ("attack.mitre.org", "github.com", "nist.gov", "dni.gov", "cia.gov", "gov.br", "planalto.gov.br")
ROTULOS = ("SYNTHETIC", "SINTÉTICO", "FICTÍCIO", "FICTICIO")
NOMES_SUSPEITOS = re.compile(r"\b(perfil|conta|instagram|linkedin)\b.{0,30}\b[A-Z][a-z]+ (de |da |do )?[A-Z][a-z]+", re.I)

def checar(pasta):
    problemas = 0
    for p in sorted(Path(pasta).rglob("*")):
        if p.suffix not in (".md", ".csv"):
            continue
        texto = p.read_text(encoding="utf-8")
        rel = p.relative_to(pasta)
        # Checagem 2: rótulo de sintético (CSV herda o rótulo do README da pasta, mas avisamos)
        if not any(r in texto.upper() for r in ROTULOS):
            print(f"[2] {rel}: sem rótulo de sintético no arquivo"); problemas += 1
        # Checagem 3: indicadores não desarmados
        for n, linha in enumerate(texto.splitlines(), 1):
            for m in PADRAO_GREP.finditer(linha):
                trecho = linha[max(0, m.start()-5): m.end()+40]
                if any(ref in trecho for ref in REFERENCIAS_OK):
                    continue                                        # link de referência: tudo bem
                print(f"[3] {rel}:{n}: «{m.group(0)}» em: {linha.strip()[:70]}"); problemas += 1
        # Checagem 4: menção a perfil de pessoa (heurística: revise à mão!)
        for m in NOMES_SUSPEITOS.finditer(texto):
            print(f"[4] {rel}: possível pessoa nomeada: «{m.group(0)}»"); problemas += 1
    print(f"\n{problemas} problema(s). A checagem 4 é heurística: confira cada caso com os próprios olhos.")
    return problemas

checar(PORT)

Corrija os três erros na célula abaixo (desarme o IP, rotule o CSV, corte a pessoa para "o padrão") e rode a checagem de novo até dar **0**.

In [ ]:
(PORT / "capstone-castellan-bec/indicators.csv").write_text(
    "# SYNTHETIC TRAINING DATA\nindicator,type\n198.51.100[.]23,ip\ncastellan-hardwood[.]example,domain\n", encoding="utf-8")
(PORT / "fraud-ring-graph/README.md").write_text(
    "# Fraud ring graph (synthetic)\n\nSYNTHETIC. Padrão de recrutamento de contas de passagem; nenhuma pessoa nomeada.\n", encoding="utf-8")
checar(PORT)

## 3. Checagem 5: as revisões escondidas de um PDF

Criamos um PDF mínimo, válido, com `/Author (Fulano Exemplo)` e um caminho local. Depois simulamos o que o `exiftool` faz: **acrescenta** uma revisão com o autor apagado. O arquivo "parece" limpo para quem lê só a última revisão, mas o autor antigo continua lá.

In [ ]:
def pdf_minimo(autor, titulo):
    """Monta um PDF de 1 página só com bytes (sem bibliotecas). Os offsets do xref são calculados."""
    objs = [b"<< /Type /Catalog /Pages 2 0 R >>",
            b"<< /Type /Pages /Kids [3 0 R] /Count 1 >>",
            b"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 200 200] >>",
            f"<< /Author ({autor}) /Title ({titulo}) /Creator (C:/Users/fulano/relatorio.docx) >>".encode()]
    saida = b"%PDF-1.7\n"
    offsets = []
    for i, o in enumerate(objs, 1):
        offsets.append(len(saida))
        saida += f"{i} 0 obj\n".encode() + o + b"\nendobj\n"
    xref = len(saida)
    saida += f"xref\n0 {len(objs)+1}\n0000000000 65535 f \n".encode()
    saida += b"".join(f"{off:010d} 00000 n \n".encode() for off in offsets)
    saida += f"trailer\n<< /Size {len(objs)+1} /Root 1 0 R /Info 4 0 R >>\nstartxref\n{xref}\n%%EOF\n".encode()
    return saida

original = pdf_minimo("Fulano Exemplo", "Relatorio capstone")
# Simula a edição incremental: um NOVO objeto Info vazio é ACRESCENTADO no fim do arquivo
revisao = original + b"5 0 obj\n<< /Author () /Title () /Creator () >>\nendobj\ntrailer\n<< /Info 5 0 R >>\n%%EOF\n"
Path("report.pdf").write_bytes(revisao)

def inspecionar(caminho):
    dados = Path(caminho).read_bytes()
    print(f"{caminho}: {dados.count(b'%%EOF')} marcador(es) %%EOF (mais de 1 = revisões acumuladas)")
    for campo in (b"/Author", b"/Creator", b"/Title"):
        for m in re.finditer(re.escape(campo) + rb"\s*\(([^)]*)\)", dados):
            if m.group(1):
                print(f"   {campo.decode()} = {m.group(1).decode(errors='replace')}")

inspecionar("report.pdf")
print("\nPor isso o original manda reescrever com o qpdf e rodar o exiftool de novo.")

In [ ]:
# Células opcionais: no Colab, com as ferramentas reais (o testador pula linhas com "!")
!apt-get -qq install -y exiftool qpdf > /dev/null
!exiftool -a -G1 -s report.pdf
!qpdf --linearize report.pdf report-clean.pdf && exiftool -a -G1 -s report-clean.pdf

## 4. 🎮 Tópicos de currículo: ação + ferramenta + resultado

O verificador abaixo procura os três elementos. Digite um tópico seu; se deixar em branco, ele avalia os exemplos do original.

In [ ]:
VERBOS = r"\b(investiguei|fundi|atribuí|pivotei|documentei|rejeitei|construí|montei|entreguei|ancorei|pontuei|" \
         r"investigated|merged|attributed|pivoted|documented|rejected|built|delivered|anchored|scored)\b"
FERRAMENTAS = r"\b(timesketch|gephi|maltego|att&ck|navigator|pandoc|python|cyberchef|zoneinfo|wireshark|tshark|sleuth|volatility|jq|exiftool)\b"
RESULTADO = r"\b(\d+|confirmad|entreguei|delivered|three|três|relatório|report)\b"
VAGO = r"\b(especialista|expert|domínio completo|experiência com|experienced with|completed|concluí)\b"

def avaliar(topico):
    tem = {"ação": re.search(VERBOS, topico, re.I), "ferramenta": re.search(FERRAMENTAS, topico, re.I),
           "resultado": re.search(RESULTADO, topico, re.I), "link": re.search(r"\[link\]|https?://|github", topico, re.I)}
    falta = [k for k, v in tem.items() if not v]
    vago = re.findall(VAGO, topico, re.I)
    print(f"«{topico[:90]}...»" if len(topico) > 90 else f"«{topico}»")
    print("   ✅ completo" if not falta and not vago else f"   falta: {falta}  palavras vagas: {vago}")

MODO_JOGO = True
meu = input("Seu tópico de currículo (Enter para usar os exemplos): ").strip() if MODO_JOGO else ""
exemplos = [
    "Concluí um programa de 90 dias de investigação cibernética cobrindo OSINT, CTI, forense e fraude.",
    "Experiência com Maltego e inteligência de ameaças.",
    "Investiguei de ponta a ponta um BEC sintético: fundi três logs numa linha do tempo UTC (Timesketch), "
    "atribuí 17 sessões e entreguei relatório com grafo de 40 nós (Gephi). [link]",
]
for t in ([meu] if len(meu) > 15 else []) + exemplos:
    avaliar(t)

## 5. O rastreador de alvos

Preencha as linhas (o exemplo traz anúncios **fictícios** só para mostrar o formato; os seus dez precisam ser reais). A célula conta as lacunas e mostra as três mais frequentes.

In [ ]:
import csv
from collections import Counter

ALVOS = [  # empregador, função, link, data, habilidades (;), item do portfólio por habilidade (;), certificação, versão, status
    ("MSSP Exemplo", "Analista de CTI", "(link real)", "2026-10-12", "ATT&CK;pivô de infraestrutura;Python;inglês técnico",
     "capstone-castellan-bec;infrastructure-pivot;lacuna;lacuna", "nenhuma", "CTI", "não candidatado"),
    ("Banco Exemplo", "Analista de prevenção a fraudes", "(link real)", "2026-10-12", "tipologias de fraude;SQL;Pix/MED",
     "fraud-ring-graph;lacuna;operacao-isca-sintetica", "CFE (desejável)", "Fraude", "não candidatado"),
    ("Consultoria Exemplo", "Analista de investigação digital", "(link real)", "2026-10-12", "OSINT;relatórios;LGPD;SQL",
     "operacao-isca-sintetica;capstone-castellan-bec;lacuna;lacuna", "nenhuma", "OSINT", "não candidatado"),
]
lacunas = Counter()
for e, f, l, d, habs, itens, cert, v, s in ALVOS:
    for h, it in zip(habs.split(";"), itens.split(";")):
        if it == "lacuna":
            lacunas[h] += 1
print("Três lacunas mais frequentes:", lacunas.most_common(3))

with open("targets.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["# PLANO: " + "; ".join(f"{h} -> (laboratório/certificação/comunidade + data)" for h, _ in lacunas.most_common(3))])
    w.writerow(["empregador", "função", "link", "data", "habilidades exigidas", "item do portfólio", "certificação", "versão do currículo", "status"])
    w.writerows(ALVOS)
print("targets.csv gravado (abre no Calc ou no Planilhas).")

## 6. 🎮 Quiz: qual artefato abre o currículo para qual função?

A tabela da Parte 2 do original. Responda o número da função.

In [ ]:
FUNCOES = ["Analista de SOC", "Analista de CTI", "Analista de OSINT", "Examinador forense", "Investigador de fraude"]
PERGUNTAS = [
    ("Linhas do tempo dos Dias 58 e 86, atribuição de sessões", 1),
    ("Grafos de pivô dos Dias 40 e 84, camadas ATT&CK", 2),
    ("Grafo do Dia 22, registros de recon, regras de engajamento", 3),
    ("Registro de evidências do Dia 83, verificação de hash", 4),
    ("Triagem e raias do Dia 87, cartão de padrão", 5),
]
for i, f in enumerate(FUNCOES, 1):
    print(f"  {i}. {f}")
acertos = 0
for artefato, certo in PERGUNTAS:
    r = input(f"\n«{artefato}» abre o currículo de qual função? ").strip() if MODO_JOGO else str(certo)
    ok = r == str(certo); acertos += ok
    print("  ✅" if ok else f"  ❌ era {certo}. {FUNCOES[certo-1]}")
print(f"\n🏁 {acertos}/{len(PERGUNTAS)}  (lembre: o grafo e o relatório dos Dias 88-89 servem a CTI, forense e fraude)")

## 7. 🎬 Fechamento da Operação Isca

| Dia | Capítulo |
|---|---|
| 83 | O cofre de evidências |
| 84 | Nove pivôs, três verdadeiros |
| 85 | As cinco perguntas da Marta |
| 86 | O relógio que ninguém acertou |
| 87 | "Não ligue para a agência" |
| 88 | A parede vira um arquivo |
| 89 | O rascunho que o Dr. Hélio devolveu |
| 90 | O que fica do caso |

No Dia 1, você escreveu `notas/minha-linha.md`: *o que me faria parar, mesmo que tecnicamente eu pudesse continuar?* Releia e responda abaixo.

In [ ]:
#@title Minha linha, 89 dias depois { display-mode: "form" }
minha_linha_dia1 = "(cole aqui o que você escreveu no Dia 1)"  #@param {type:"string"}
o_que_mudou = "Continuo parando diante de credenciais alheias e de pessoas no lugar de padrões. Aprendi a parar também diante da minha própria certeza: se não tenho duas fontes, escrevo provável."  #@param {type:"string"}

from datetime import datetime, timezone
texto = (f"# Minha linha — revisão do Dia 90\n\nDia 1: {minha_linha_dia1}\n\nDia 90 ({datetime.now(timezone.utc):%Y-%m-%d}): {o_que_mudou}\n")
Path("notas").mkdir(exist_ok=True)
Path("notas/minha-linha-dia90.md").write_text(texto, encoding="utf-8")
print(texto)
try:
    from google.colab import files
    files.download("notas/minha-linha-dia90.md"); files.download("targets.csv")
except ImportError:
    print("(Fora do Colab: notas/minha-linha-dia90.md e targets.csv)")

## ✅ Antes de fechar o Dia 90 (e os 90 dias)

- [ ] Portfólio público: grafo do capstone a dois cliques do README
- [ ] Checagens 1 a 5 passando em todo artefato (incluindo o PDF reescrito)
- [ ] Currículo de uma página; todo tópico com ação, ferramenta, resultado e link
- [ ] Rastreador com dez anúncios reais (cinco brasileiros), três lacunas com plano e data
- [ ] Quadro do Dia 1 reconciliado
- [ ] `minha-linha.md` relido e respondido

🥷 **Rank S.** Obrigado por ter feito o caminho inteiro.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>